# Train Arm D (blt_bpe_guided) on Kaggle - full pipeline test

See `plans/PLAN.md` (question 1.3), `phases/phase-2-small-train.md` ("Hướng cải thiện BLT
sau khi có kết luận 1.3"). Arm D: vẫn byte-level (vocab=256) nhưng dùng tokenizer BPE thật
(PhoGPT-4B) chỉ để tìm ranh giới patch, không dùng vocab/embedding của nó (ý tưởng từ paper
Super Tiny Language Models, arXiv:2405.14159) - không có entropy model nên bỏ hẳn bước
pretrain entropy. Same GPU-detect + torch pin + dynamic CODE_DIR as the other Kaggle
notebooks in this repo.


In [ ]:
import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_name or "(none detected)")

if "P100" in gpu_name:
    print("P100 detected - pinning torch==2.7.1+cu126 (last version supporting sm_60)")
    subprocess.run(
        ["pip", "install", "-q", "torch==2.7.1", "--index-url",
         "https://download.pytorch.org/whl/cu126"],
        check=True,
    )
else:
    print("Not a P100 - keeping the pre-installed torch build")

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
import os

_candidates = [
    "/kaggle/input/vislm-research-code",
    "/kaggle/input/datasets/nguyennn263/vislm-research-code",
]
CODE_DIR = next(p for p in _candidates if os.path.isdir(p))
os.environ["PYTHONPATH"] = CODE_DIR
print("CODE_DIR:", CODE_DIR)

!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python {CODE_DIR}/setup/download_prepare_data.py \
  --target-gb 0.05 --out-dir /kaggle/working/data/prepared/fineweb2_vi


In [ ]:
# First real run of Arm D (bpe_guided boundaries) - small scale to verify the pipeline
# end-to-end (forward/backward already verified locally) before committing to a bigger run.
# No entropy_pretrain_steps override - Arm D has no entropy model, train.py skips that
# step automatically.
!python -m vislm.train {CODE_DIR}/pillar1_configs/1_3_arm_D_bpe_guided.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_D_debug \
  model.max_seq_len=128 train.batch_size=4 \
  train.max_steps=30


In [ ]:
import json

losses = []
with open("/kaggle/working/runs/arm_D_debug/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            losses.append(row["loss"])

summary = {
    "n_steps": len(losses),
    "first_loss": losses[0],
    "last_loss": losses[-1],
    "min_loss": min(losses),
}
print(summary)

with open("/kaggle/working/metrics_train_arm_d_debug.jsonl", "w") as f:
    f.write(json.dumps({"section": "train_arm_d_debug", "results": summary}) + "\n")
